# 08. 신호는 얼마나 맞히고 놓치나, 그리고 기준 기간을 공유하지 않는 신호로 다시 보기
> 계획서: `계획/P9_보강검증_적중과_기준기간.md` · 코드: `code/hit_cnt.py` · 전체 계산은 P5·P7 과 같은 코드(`validate_cnt`, `robust_cnt`)를 그대로 씁니다.

발표에서 받을 질문 두 가지에 답하려고 만든 **보강 점검**입니다. (규칙·기준은 바꾸지 않습니다. 표 위주이고 새로운 통계 기법은 없습니다.)

| 점검 | 질문 | 어떻게 |
|---|---|---|
| **A. 적중과 놓침** | 신호가 잡힌 기업 중 몇 곳이 실제로 요구불입금액이 약해졌고, 약해진 기업 중 몇 곳을 신호가 잡았나? | 신호/비신호 × 약화/비약화 **2×2 표**(P5의 RR과 같은 기업-월 단위), 감소 기준(n)별 변화, 기업 단위로 본 신호의 시점 |
| **B. 기준 기간 비공유** | 신호의 비교 달과 약화의 기준 달이 같아서 둘이 함께 움직이는 것 아닌가? | 신호를 **전년 같은 달 대비 감소**로 정의해(비교 달이 약화의 기준 달과 겹치지 않음) 같은 분석을 다시 함 |

**읽는 법**: A는 점수표이므로 판정 문턱이 없습니다. B는 P5와 같은 사전 규칙(RR 95% 범위의 하한 > 1 이면 연관 유지)으로 읽고, 기준 규칙과의 차이는 '기준 대비 비'로 봅니다. 화면 출력은 집계값뿐 · 5 미만 칸은 `<5` · 법인ID 없음. 몇 분 걸립니다(`CNT_BOOT` 로 줄일 수 있음).

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "건수분석" / "code", base / "yangdaecheon" / "건수분석" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 yangdaecheon/건수분석/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
pops = {"P36": masks["P36"], "P35+": masks["P35p"]}
print({k: int(v.sum()) for k, v in pops.items()}, "개 기업")

import math
import detect_cnt as dc
import validate_cnt as vc
import robust_cnt as rb
import hit_cnt as ht

spec = json.load(open(WORK / "interim" / "detect_spec_v2.json", encoding="utf-8"))
rule = spec["rule"]
MODE, K, L, PHI2, N2 = rule["비교 기준"], int(rule["k"]), int(rule["L"]), float(rule["phi"]), float(rule["n_pct"])
B = int(os.environ.get("CNT_BOOT", "2000"))
SEED = 20261301           # 기준과 같은 시드(= 같은 기업 재표집)로 짝지어 비교한다

def sup(x):
    return cl.sup(int(round(x)))

def pct(x, d=1):
    return None if x is None or not np.isfinite(x) else round(100 * x, d)

def f_rr(rr, lo, hi):
    return "-" if not np.isfinite(rr) else (f"{rr:.2f} ({lo:.2f}~{hi:.2f})" if np.isfinite(lo) else f"{rr:.2f}")

mk36, mk35 = masks["P36"], masks["P35"]
C5 = vc.fit_constants(panel, mk36, MODE, K, PHI2, N2)
D36 = vc.prepare(panel, mk36, C5)
assert int((D36["det"] >= 0).sum()) == spec["power"]["P36"]["N1"], "04의 탐지 기업 수와 다릅니다 (04를 다시 실행하세요)"
F36, ER = D36["F"], D36["er"]
BASE = rb.run_variant(D36, F36, B, SEED)
BASE_OBJ = {"R": BASE["R"], "D": D36}
Dh = vc.prepare(panel, mk35, C5)
print(f"기준(P36, v2): 탐지 {BASE['n_det']:,}곳, RR {f_rr(BASE['R']['rr'], BASE['R']['lo'], BASE['R']['hi'])}  /  기준월 {len(ER)}개 (e={ER[0]}~{ER[-1]})")

## A-1. 신호/비신호 × 약화/비약화 (2×2 표)
같은 기업-월 단위에서 세어 봅니다. **신호** = 그 기준월에 처음 신호가 잡힌 기업, **비신호** = 같은 기준월에 판정 가능하지만 아직 신호가 없던 기업(P5의 비교 기업과 같은 모집단). **약화** = 그 기준월 이후 6개월 요구불입금액이 전년 같은 시기보다 기준(하위 5%) 이상 줄어든 경우입니다.

- **적중 비율** = 신호 기업-월 중 약화 비율(P5의 24.6%와 같아야 합니다).
- **약화 기업-월 중 신호가 잡은 비율** = 약화 사건 가운데 *그 달에* 신호가 있었던 비율. 이 단위는 같은 기업이 여러 기준월에 나올 수 있으니 기업 단위 표(A-2)와 함께 보세요.

In [ ]:
U36, Uh = D36["units"], Dh["units"]
T36, Th = ht.two_by_two(U36), ht.two_by_two(Uh)
S36 = vc.estimate(U36, *vc.cells_main(U36))
Sh = vc.estimate(Uh, *vc.cells_main(Uh))
if S36["excluded_share"] == 0:
    assert abs(T36["precision"] - BASE["R"]["p_det"]) < 1e-9, "적중 비율이 P5의 탐지군 약화율과 다릅니다"
rows, rows2 = [], []
for nm, T, S, D in (("P36 (주)", T36, S36, D36), ("P35만 (독립 기업)", Th, Sh, Dh)):
    M = len(D["er"])
    mo = ht.monthly_averages(T, M)
    rows += [{"표본": nm, "구분": "신호 기업-월", "약화": sup(T["a"]), "비약화": sup(T["b"]), "합계": sup(T["n_signal"]), "약화 비율 %": pct(T["precision"])},
             {"표본": nm, "구분": "비신호 기업-월 (위험집합)", "약화": sup(T["c"]), "비약화": sup(T["d"]), "합계": sup(T["n_other"]), "약화 비율 %": pct(T["ctrl_rate"])}]
    rows2 += [{"표본": nm, "지표": "적중 비율 (신호 기업-월 중 약화) %", "값": pct(T["precision"])},
              {"표본": nm, "지표": "비신호 기업-월의 약화 비율 % (조정 전)", "값": pct(T["ctrl_rate"])},
              {"표본": nm, "지표": "비교 기업의 약화 비율 % (층화·표준화, P5와 같음)", "값": pct(S["p_ctrl"])},
              {"표본": nm, "지표": "RR (표준화 기준, 점추정)", "값": round(S["rr"], 2)},
              {"표본": nm, "지표": "약화 기업-월 중 신호가 그 달에 잡은 비율 %", "값": pct(T["recall"])},
              {"표본": nm, "지표": "약화 기업-월 중 신호가 없던 비율 % (놓침)", "값": pct(T["miss_share"])},
              {"표본": nm, "지표": f"월평균 신호 기업 (기준월 {M}개)", "값": round(mo["signals_per_month"], 1)},
              {"표본": nm, "지표": "월평균 약화 기업-월 (신호 + 비신호)", "값": round(mo["weak_per_month"], 1)},
              {"표본": nm, "지표": "그중 월평균 신호가 잡은 수", "값": round(mo["caught_per_month"], 1)}]
cl.show_table(pd.DataFrame(rows), "신호 × 약화 2×2 표 (기업-월)", OUT / "nb08", "two_by_two")
cl.show_table(pd.DataFrame(rows2), "적중과 놓침 요약", OUT / "nb08", "hit_summary")

## A-2. 기업 단위로 보면 — 신호가 약화보다 먼저였나
기업 단위 요약입니다. **약화 기업** = 판정 가능했던 기준월 중 **한 번이라도** 약화가 있었던 기업이고, 그 첫 약화 기준월보다 신호가 **같거나 먼저** 잡혔는지 봅니다. (신호가 먼저 잡혔다고 해서 신호가 입금 약화를 *일으켰다는* 뜻은 아닙니다. 두 정의가 같은 기준 기간을 쓰므로 B에서 다시 확인합니다.)

In [ ]:
FT36, FTh = ht.firm_timing(D36), ht.firm_timing(Dh)
rows = []
for nm, FT in (("P36 (주)", FT36), ("P35만 (독립 기업)", FTh)):
    nj, nw = FT["n_judgeable"], FT["n_weak"]
    rows += [{"표본": nm, "구분": "판정 가능 기업 (기준월 중 한 번이라도)", "기업 수": sup(nj), "비율 %": 100.0, "분모": "판정 가능 기업"},
             {"표본": nm, "구분": "그중 약화 기업 (한 번이라도 약화)", "기업 수": sup(nw), "비율 %": pct(nw / nj), "분모": "판정 가능 기업"},
             {"표본": nm, "구분": "약화 기업 중 신호가 먼저 (또는 같은 달)", "기업 수": sup(FT["early"]), "비율 %": pct(FT["early"] / nw), "분모": "약화 기업"},
             {"표본": nm, "구분": "약화 기업 중 신호가 더 늦음", "기업 수": sup(FT["late"]), "비율 %": pct(FT["late"] / nw), "분모": "약화 기업"},
             {"표본": nm, "구분": "약화 기업 중 신호가 한 번도 없음 (놓침)", "기업 수": sup(FT["none"]), "비율 %": pct(FT["none"] / nw), "분모": "약화 기업"},
             {"표본": nm, "구분": "신호 기업 (참고)", "기업 수": sup(FT["n_signal"]), "비율 %": pct(FT["n_signal"] / nj), "분모": "판정 가능 기업"},
             {"표본": nm, "구분": "신호 기업 중 약화 기업 (참고)", "기업 수": sup(FT["n_signal_weak"]), "비율 %": pct(FT["n_signal_weak"] / max(FT["n_signal"], 1)), "분모": "신호 기업"}]
cl.show_table(pd.DataFrame(rows), "기업 단위: 약화 기업과 신호의 시점", OUT / "nb08", "firm_timing")

## A-3. 감소 기준(n)을 바꾸면 — 적중과 놓침의 맞교환
감소 기준을 높이면 신호는 줄고 적중 비율은 올라가지만, 약화를 놓치는 비율도 커집니다. 다른 모든 규칙·문턱(Φ·비교 기준·k·θ·경계)은 P36에서 고정한 값 그대로입니다. 인력이 부족할 때 어느 기준에서 어느 정도를 잡는지 보여 주는 표입니다.

In [ ]:
NS = [20, 25, 30, 35, 40, 45, 50, 60, 70]
GRID = ht.precision_recall_grid(panel, mk36, C5, NS)
rows = [{"감소 기준 n %": int(g["n"]) , "신호 기업(최초)": sup(g["n_detected"]), "월평균 신호 기업": round(g["n_detected"] / g["n_months"], 1), "적중 비율 %": pct(g["precision"]),
         "비교 기업 약화 비율 % (표준화)": pct(g["p_ctrl_std"]), "RR (점추정)": round(g["rr"], 2), "약화 기업-월 중 신호가 잡은 비율 %": pct(g["recall"]),
         "월평균 잡은 약화 기업-월": round(g["a"] / g["n_months"], 1), "월평균 약화 기업-월 (전체)": round((g["a"] + g["c"]) / g["n_months"], 1)} for g in GRID]
TAB_GRID = pd.DataFrame(rows)
cl.show_table(TAB_GRID, "감소 기준별 적중과 놓침 (P36, 기업-월 단위)", OUT / "nb08", "n_grid")
g35 = [g for g in GRID if int(g["n"]) == int(N2)]
if g35:
    assert abs(g35[0]["precision"] - T36["precision"]) < 1e-9, "n=35% 행이 A-1 과 다릅니다"
fig, ax = plt.subplots(figsize=(6.5, 4.4))
xs, ys = [100 * g["recall"] for g in GRID], [100 * g["precision"] for g in GRID]
ax.plot(xs, ys, "-o", color="#2a78d6", lw=2)
for g, x, y in zip(GRID, xs, ys):
    ax.annotate(f"{int(g['n'])}%", (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.axhline(100 * S36["p_ctrl"], color="#666666", ls="--", lw=1)
ax.text(max(xs) * 0.98, 100 * S36["p_ctrl"] + 1.2, "비교 기업의 약화 비율(표준화)", ha="right", fontsize=8, color="#444444")
ax.set_xlabel("약화 기업-월 중 신호가 잡은 비율 % (오른쪽일수록 덜 놓침)")
ax.set_ylabel("적중 비율 % (위쪽일수록 맞는 신호)")
ax.set_title("감소 기준을 올리면 적중은 늘고 놓침도 는다 (P36, 점 옆 = 감소 기준)")
plt.tight_layout(); plt.show()

## B. 기준 기간을 공유하지 않는 신호
기준 규칙(직전 6개월 대비 감소)에서는 **신호의 비교 달이 요구불입금액 약화의 기준 달과 같습니다**(둘 다 e−11~e−6). 그래서 그 기간에 건수와 요구불입금액이 함께 높았다면 평소로 돌아오기만 해도 둘이 함께 줄어 보일 수 있습니다(기저효과).
**신호를 '전년 같은 달 대비 감소'로 정의하면 신호의 비교 달이 e−17~e−12로 옮겨 가 약화의 기준 달과 겹치지 않습니다.** 이 규칙에서 활동량 하한 Φ와 감소 기준 n은 P4와 같은 절차로 다시 구합니다(건수만 사용, 요구불입금액은 쓰지 않음). 약화의 정의와 θ는 P5와 같은 방식입니다.

In [ ]:
rows = [{"신호 정의": nm, "비교 달 (기준월 e 기준)": f"e{r['signal_reference'][0] - 20:+d} ~ e{r['signal_reference'][1] - 20:+d}", "약화의 기준 달": f"e{r['outcome_base'][0] - 20:+d} ~ e{r['outcome_base'][1] - 20:+d}",
         "겹치는 달 수": r["overlap_months"]} for nm, r in (("직전 기간 대비 (기준 규칙)", ht.reference_overlap(K, dc.lag_for("직전", K))), ("전년 같은 달 대비 (이번 점검)", ht.reference_overlap(K, 12)))]
cl.show_table(pd.DataFrame(rows), "신호의 비교 달과 약화의 기준 달", OUT / "nb08", "reference_months")
d_y, C_y = ht.year_on_year_rule(panel, mk36, K)
D_y = vc.prepare(panel, mk36, C_y)
rows = [{"규칙": "기준 규칙 (직전 기간 대비)", "비교 기준": MODE, "k": K, "활동량 하한 Φ": round(PHI2, 2), "n (%)": N2, "탐지 기업": sup(int((D36["det"] >= 0).sum())), "판정 가능 기업": sup(int((D36["status"][:, ER] == 0).any(axis=1).sum()))},
        {"규칙": "이번 점검 (전년 같은 달 대비)", "비교 기준": "전년", "k": K, "활동량 하한 Φ": round(d_y["phi"], 2), "n (%)": d_y["n"], "탐지 기업": sup(int((D_y["det"] >= 0).sum())), "판정 가능 기업": sup(int((D_y["status"][:, D_y["er"]] == 0).any(axis=1).sum()))}]
cl.show_table(pd.DataFrame(rows), "두 신호 규칙 (건수만으로 정한 값)", OUT / "nb08", "rules")

In [ ]:
t0 = time.time()
RY = rb.run_variant(D_y, F36, B, SEED, BASE_OBJ)
RH = rb.run_variant(Dh, Dh["F"], B, 20261302)
Dh_y = vc.prepare(panel, mk35, C_y)
RHY = rb.run_variant(Dh_y, Dh_y["F"], B, 20261302, {"R": RH["R"], "D": Dh})
print(f"계산 완료 ({time.time() - t0:.0f}초)")
rows = []
for nm, res in (("P36 · 기준 규칙 (직전 기간 대비)", {"R": BASE["R"], "n_det": BASE["n_det"], "descriptive": False}), ("P36 · 전년 같은 달 대비 (비교 달 비공유)", RY),
                ("P35만 · 기준 규칙", RH), ("P35만 · 전년 같은 달 대비 (비교 달 비공유)", RHY)):
    R = res["R"]
    ovl = res.get("overlap")
    rows.append({"표본·신호 정의": nm, "신호 기업": sup(res["n_det"]), "신호 기업 약화율 %": pct(R["p_det"]), "비교 기업 약화율 % (표준화)": pct(R["p_ctrl"]), "RR (95% 범위)": f_rr(R["rr"], R["lo"], R["hi"]),
                 "기준 대비 비 (95% 범위)": f_rr(*res["ratio"]) if "ratio" in res else "-", "최초 신호 일치(Jaccard)": round(ovl["jaccard"], 2) if ovl else None,
                 "읽는 법": "연관 유지 (범위 하한 > 1)" if (np.isfinite(R["lo"]) and R["lo"] > 1) else "범위가 1을 포함하거나 정의 불가"})
TAB_B = pd.DataFrame(rows)
cl.show_table(TAB_B, "기준 기간을 공유하지 않는 신호의 RR", OUT / "nb08", "year_on_year")
ry = RY["R"]
if np.isfinite(ry["lo"]) and ry["lo"] > 1:
    print(f"전년 같은 달 대비 신호에서도 RR {ry['rr']:.2f} (95% 범위 {ry['lo']:.2f}~{ry['hi']:.2f}), 하한 > 1 → 기준 달 공유만으로는 연관이 설명되지 않음 (해석은 Claude 와 함께)")
else:
    print(f"전년 같은 달 대비 신호의 RR {ry['rr']:.2f} (95% 범위 {ry['lo']:.2f}~{ry['hi']:.2f}) 가 1을 포함 → 기저효과 가능성이 남음")
T_y = ht.two_by_two(D_y["units"])
rows = [{"신호 정의": nm, "적중 비율 %": pct(T["precision"]), "비신호 약화 비율 % (조정 전)": pct(T["ctrl_rate"]), "약화 기업-월 중 신호가 잡은 비율 %": pct(T["recall"]), "월평균 신호 기업": round(T["n_signal"] / len(Dd["er"]), 1)}
        for nm, T, Dd in (("직전 기간 대비 (기준 규칙)", T36, D36), ("전년 같은 달 대비", T_y, D_y))]
cl.show_table(pd.DataFrame(rows), "두 신호 규칙의 적중과 놓침 (P36, 기업-월)", OUT / "nb08", "two_rules_hit")

## 요약 저장

In [ ]:
def pk(res):
    R = res["R"]
    d = {k: (None if not np.isfinite(R[k]) else float(R[k])) for k in ("rr", "lo", "hi", "p_det", "p_ctrl")}
    d["n_det"] = int(res["n_det"])
    d["ratio"] = None if "ratio" not in res else [float(x) for x in res["ratio"]]
    d["jaccard"] = None if not res.get("overlap") else float(res["overlap"]["jaccard"])
    return d

def clean(d):
    return {k: (None if isinstance(v, float) and not np.isfinite(v) else v) for k, v in d.items()}

summary = {"A": {"two_by_two": {"P36": clean(T36), "P35": clean(Th)}, "std_ctrl": {"P36": float(S36["p_ctrl"]), "P35": float(Sh["p_ctrl"])},
                 "firm_timing": {"P36": FT36, "P35": FTh}, "grid": [clean({k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in g.items()}) for g in GRID]},
           "B": {"overlap": {"prior": ht.reference_overlap(K, dc.lag_for("직전", K)), "yoy": ht.reference_overlap(K, 12)}, "yoy_rule": {"phi": float(d_y["phi"]), "n": float(d_y["n"])},
                 "P36_base": pk({"R": BASE["R"], "n_det": BASE["n_det"]}), "P36_yoy": pk(RY), "P35_base": pk(RH), "P35_yoy": pk(RHY), "two_by_two_yoy": clean(T_y)},
           "synthetic": bool(panel.synthetic)}
cl.save_json(WORK / "interim" / "nb08_summary.json", summary)
print("저장: interim/nb08_summary.json,", "output/nb08/*.csv")

## 마치면
실행 후 **저장(Ctrl+S)** 만 해 주세요. 결과 표는 `data/work_cnt/output/nb08/`에 자동 저장되어 Claude가 직접 읽고 해석합니다.